# Paper figures

Every figure of the paper that rests on device data, redrawn from the data in this repository. Each
section says which runs it uses, where they are stored (`data/results/<backend>/<structure>/<kind>/`), and
how they were analysed. The PDFs go to `figures/paper_figures/`.

All data is in this package's result format, whether it was taken with this package or converted from
earlier campaigns by `scripts/import_legacy_chains.py`. For every result, $r$ and its shot-noise error
are recomputed from the stored samples, the noiseless reference $r_{\rm ideal}$ is exact, and the
random baseline is exact: $r_{\rm rand} = 1/2$, with shot-noise spread $\sigma_{\rm rand} = 1/(2\sqrt{(n-1)S})$
for $S$ shots. The runs are named explicitly, so adding a campaign to `data/` never changes a published
figure by accident.

| figure | what it shows | section |
|---|---|---|
| Fig. 3c | distribution of $r$ over every data-ancilla-data triplet, per device, at $p = 3$ | 1 |
| Fig. 3d | median $r$ (bars: inter-quartile range) over the triplets, against depth | 2 |
| Fig. 3e | $r$ against depth for each device's best triplet, with Helios-1E | 3 |
| Fig. 3f | the same triplets as $r_{\rm ovl}$, with points indistinguishable from random marked | 4 |
| Fig. 4 | effective error per edge $\lambda_{\rm eff}$ against chain length: Helios-1 (serialized runs up to 25 August, parallel from 9 September) and IBM Boston | 5 |
| Fig. 5a | depolarizing simulation: overlap collapse per chain length, $\kappa_r = \kappa_0/N_q$ | 6 |
| Fig. 5b | $\lambda_{\rm eff}$ of a 10-spin chain on IBM Boston and the Helios-1E emulator, direct and MCM | 6 |
| Fig. 5c | $\lambda_{\rm eff}$ against $N_q$ for eight devices, direct and MCM | 6 |
| Fig. 5d | mean MCM $\lambda_{\rm eff}$ of eight devices against their release date | 6 |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
RESULTS = ROOT / "data" / "results"
FIGURES = ROOT / "figures" / "paper_figures"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qecbench import Chain
from qecbench.analysis import load_results
from qecbench.lrqaoa import ideal_r, random_baseline

mpl.rcParams.update({
    "xtick.labelsize": 16, "ytick.labelsize": 16, "font.size": 16, "axes.linewidth": 2,
    "axes.titlesize": 16, "axes.labelsize": 16, "lines.linewidth": 2.2, "lines.markersize": 3,
    "lines.markeredgewidth": 1.2, "errorbar.capsize": 3, "legend.fontsize": 14,
    "xtick.major.width": 3, "xtick.major.size": 10, "ytick.major.width": 3, "ytick.major.size": 8,
    "xtick.minor.width": 1.5, "xtick.minor.size": 6, "ytick.minor.width": 1.5, "ytick.minor.size": 6,
    "mathtext.fontset": "dejavusans", "font.family": "sans-serif",
})
%matplotlib inline
SHOW_LABELS = False                    # the paper panels have no axis labels or legends; True adds them


def runs(backend, stamps, kind="mcm", structure="chain", **kwargs):
    # results of the named run files only: <stamp>_<backend>_<structure>_<kind>.json
    files = {f"{stamp}_{backend}_{structure}_{kind}.json" for stamp in stamps}
    return load_results(RESULTS, backend, kind=kind, files=files, **kwargs)


TRIPLET = Chain((0, 1, 2))             # any triplet: the references depend only on the Hamiltonian


def noiseless(p):
    return ideal_r(TRIPLET, int(p))


def random_sigma(shots):
    return random_baseline(TRIPLET, shots)[1]

### Data shared by Fig. 3c-3f

**Triplet campaigns.** One campaign per device, 500 shots per circuit, every data-ancilla-data triplet the
device offers, at $p = 3, 6, 9, 12$. Converted from the triplet files of
`Benchmarking-Mid-circuit-measurement/Data` (`<stamp>_<backend>_tri_<d1>_<a>_<d2>_mcm_nq2_depth<p>.json`) into
`data/results/<backend>/chain/mcm/`:

| device | run file(s) | triplets |
|---|---|---|
| `iqm_garnet` | `20260813_0729_iqm_garnet_chain_mcm.json` | 35 |
| `iqm_emerald` | `20260813_0722_…`, `20260813_0723_iqm_emerald_chain_mcm.json` | 104 |
| `ibm_kingston` | `20260813_1637_ibm_kingston_chain_mcm.json` | 148 |
| `ibm_boston` | `20260814_0818_ibm_boston_chain_mcm.json` | 148 |
| `ibm_phoenix` | `20260902_1629_ibm_phoenix_chain_mcm.json` | 120 |

Later campaigns on Boston (11 September) and Phoenix (9 and 11 September) are not part of these figures.

**Helios-1E two-spin runs** (Fig. 3e, 3f). The same two-spin MCM circuit on Quantinuum's **Helios-1E
emulator** (24 February 2026), 500 shots at $p = 3, 5, 7, 10, 12$, one file per depth. Its logical labels are
data qubits 0, 1 and ancilla 2. The files, in the older MaxCut schema (for one edge the cut ratio equals
$r$), are converted into `data/results/Helios-1E/chain/mcm/20260224_{153017,153157,153300,153359,154357}_Helios-1E_chain_mcm.json`
and flagged as simulated.

**One difference from the first versions of these figures.** The values of $r$, the medians, the quartiles
and the noiseless curves are unchanged. The random reference is now the exact baseline, $1/2 \pm 3 \cdot 1/(2\sqrt{500})$,
instead of a bootstrap estimate ($0.506 \pm 3 \cdot 0.020$). The grey bands are therefore centred on $1/2$ and
slightly wider.

In [ ]:
TRIPLET_RUNS = {                       # the order used in every panel
    "iqm_garnet":   ["20260813_0729"],
    "iqm_emerald":  ["20260813_0722", "20260813_0723"],
    "ibm_kingston": ["20260813_1637"],
    "ibm_boston":   ["20260814_0818"],
    "ibm_phoenix":  ["20260902_1629"],
}
HELIOS_PAIR_RUNS = {"Helios-1E": ["20260224_153017", "20260224_153157", "20260224_153300",
                                  "20260224_153359", "20260224_154357"]}
COLOURS = {"iqm_garnet": "#3b6fd4", "iqm_emerald": "#c4c43f", "ibm_kingston": "#8a4bc9",
           "ibm_boston": "#b8560f", "ibm_phoenix": "#2a9d8f", "Helios-1E": "#0f9b8e"}

# {device: {triplet: {p: summary}}}; summary holds r, r_err, r_ideal, r_ovl, r_rand, r_rand_std, shots
sweeps = {device: runs(device, stamps, kind="mcm", n_data=2) for device, stamps in {**TRIPLET_RUNS, **HELIOS_PAIR_RUNS}.items()}
for device, per_triplet in sweeps.items():
    depths = sorted({p for by in per_triplet.values() for p in by})
    shots = sorted({s["shots"] for by in per_triplet.values() for s in by.values()})
    print(f"{device:>13}: {len(per_triplet):>3} triplet(s), depths {depths}, shots {shots}")

## 1. Fig. 3c - approximation ratio of every triplet, per device

**What it shows.** Each device's mid-circuit-measurement cycle, tested on every data-ancilla-data triplet
`(d1, a, d2)` of the chip: a two-spin LR-QAOA of depth $p = 3$ whose $ZZ$ term runs through the ancilla, its
mid-circuit measurement and the feed-forward. One row per device, from top to bottom: IQM Garnet, IQM
Emerald, IBM Kingston, IBM Boston, IBM Phoenix. Each point is one triplet. The shaded violin is the
distribution, the bar under it the inter-quartile range, and the circle the median. The dashed line is the
noiseless $r_{\rm ideal}(p=3)$, and the grey band is random guessing, $r_{\rm rand} \pm 3\sigma_{\rm rand}$. A device
whose triplets sit close to the dashed line runs the cycle well on every qubit, and the width of its row
is how uneven the chip is.

In [ ]:
DEPTH_3C = 3
triplet_r = {d: np.array(sorted(by[DEPTH_3C]["r"] for by in sweeps[d].values() if DEPTH_3C in by)) for d in TRIPLET_RUNS}
S = sweeps["ibm_boston"][next(iter(sweeps["ibm_boston"]))][DEPTH_3C]["shots"]
sigma = random_sigma(S)

fig, ax = plt.subplots(figsize=(3, 5))
rng = np.random.default_rng(0)
rows = list(TRIPLET_RUNS)[::-1]        # first device on top
for y, device in enumerate(rows):
    v, colour = triplet_r[device], COLOURS[device]
    for body in ax.violinplot([v], positions=[y], vert=False, widths=0.8, showextrema=False,
                              showmedians=False)["bodies"]:
        body.set_facecolor(colour); body.set_alpha(0.25); body.set_edgecolor("none")
    ax.scatter(v, y + rng.uniform(-0.16, 0.16, v.size), s=22, color=colour, alpha=0.65, edgecolors="none", zorder=3)
    q1, med, q3 = np.quantile(v, [0.25, 0.5, 0.75])
    ax.plot([q1, q3], [y - 0.3] * 2, color=colour, lw=3, solid_capstyle="butt", zorder=4)
    ax.plot([med], [y - 0.3], marker="o", ms=10, color=colour, zorder=5, markeredgecolor="black")
ax.axvspan(0.5 - 3 * sigma, 0.5 + 3 * sigma, color="gray", alpha=0.25, lw=0, zorder=0)
ax.axvline(noiseless(DEPTH_3C), ls="--", lw=2, color="black", zorder=1)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([d.replace("_", " ") for d in rows] if SHOW_LABELS else [""] * len(rows))
ax.set_ylim(-0.7, len(rows) - 0.05)
if SHOW_LABELS:
    ax.set_xlabel("approximation ratio $r$")
fig.savefig(FIGURES / f"fig3c_mcm_r_distribution_p{DEPTH_3C}.pdf", transparent=True, bbox_inches="tight")
plt.show()

print(f"p = {DEPTH_3C}, {S} shots: r_ideal = {noiseless(DEPTH_3C):.3f}, random band 0.500 ± {3 * sigma:.3f} (3 sigma)")
for device, v in triplet_r.items():
    print(f"{device:>14}: n={len(v):>3}  median {np.median(v):.3f}  IQR [{np.quantile(v, .25):.3f}, "
          f"{np.quantile(v, .75):.3f}]  range [{v.min():.3f}, {v.max():.3f}]")

## 2. Fig. 3d - approximation ratio against depth, over all triplets

**What it shows.** For each device, the median $r$ over all its triplets at $p = 3, 6, 9, 12$, with the
inter-quartile range as error bars (a small horizontal offset keeps the devices apart). The dashed line is
the noiseless $r_{\rm ideal}(p)$, which rises with depth, and the grey band is random guessing. On a perfect
device the points would follow the dashed line upwards. The gap to it, and whether it grows with $p$, is
the accumulated cost of one more measure-and-correct cycle per layer.

In [ ]:
DEPTHS_3D = [3, 6, 9, 12]
stats = {}
for device in TRIPLET_RUNS:
    for p in DEPTHS_3D:
        v = np.array([by[p]["r"] for by in sweeps[device].values() if p in by])
        q1, mid, q3 = np.quantile(v, [0.25, 0.5, 0.75])
        stats.setdefault(device, []).append((p, mid, mid - q1, q3 - mid, len(v)))

fig, ax = plt.subplots(figsize=(3, 5))
ax.axhspan(0.5 - 3 * sigma, 0.5 + 3 * sigma, color="gray", alpha=0.25, lw=0, zorder=0, label="random $3\\sigma$")
fine = np.arange(min(DEPTHS_3D), max(DEPTHS_3D) + 1)
ax.plot(fine, [noiseless(p) for p in fine], ls="--", lw=2, color="black", zorder=1, label="noiseless")
for dx, device in zip(np.linspace(-0.35, 0.35, len(TRIPLET_RUNS)), TRIPLET_RUNS):
    p, mid, lo, hi, n = np.array(stats[device]).T
    ax.errorbar(p + dx, mid, yerr=[lo, hi], color=COLOURS[device], lw=2.2, marker="o", ms=9,
                markeredgecolor="black", capsize=5, capthick=2.2, elinewidth=2.2, zorder=3,
                label=device.replace("_", " "))
ax.set_xticks(DEPTHS_3D)
ax.set_ylim(0.4, 1.02)
if SHOW_LABELS:
    ax.set(xlabel="$p$", ylabel="approximation ratio $r$")
    ax.legend(fontsize=10, loc="upper center", bbox_to_anchor=(0.5, 1.4), ncol=2)
fig.savefig(FIGURES / "fig3d_mcm_r_vs_p_median.pdf", transparent=True, bbox_inches="tight")
plt.show()

print(f"{'device':>14} " + " ".join(f"{'p=' + str(p):>18}" for p in DEPTHS_3D))
for device in TRIPLET_RUNS:
    print(f"{device:>14} " + " ".join(f"{mid:>8.3f} -{lo:.3f}+{hi:.3f}" for _, mid, lo, hi, _ in stats[device]))

## 3. Fig. 3e - the best triplet of each device, against depth

**What it shows.** Fig. 3c-3d describe whole chips. This panel asks how good the gadget can be at all: one
curve per device for its single best triplet, and the same circuit on Quantinuum's Helios-1E emulator, where
the ancilla is allocated freely on an all-to-all machine, for scale. The dashed line is the noiseless
$r_{\rm ideal}(p)$ and the grey band random guessing.

**Which triplet is "best".** The one with the highest $r_{\rm ovl}$ at $p = 3$, among triplets measured at
three or more depths. Its whole sweep is then drawn. Choosing by one depth is a statement about that depth:
the winners sit well above their device's median, but the ranking over the full sweep can differ (on
Boston, the $p = 3$ winner is not the best triplet over all depths). The table below gives each winner's
shot-noise uncertainty and how many other triplets are within it (`ties`). Where `ties` is not zero, the
choice of winner is not resolved at 500 shots.

Devices shown: Helios-1E, IQM Garnet, IQM Emerald, IBM Kingston and IBM Boston, as in the published panel.
IBM Phoenix was measured after that panel was made; add it to `FIG3EF_DEVICES` to include it.

In [ ]:
FIG3EF_DEVICES = ["Helios-1E", "iqm_garnet", "iqm_emerald", "ibm_kingston", "ibm_boston"]
SCORE_DEPTH = 3


def score(by):
    s = by[SCORE_DEPTH]
    return s["r_ovl"], np.sqrt(s["r"] * (1 - s["r"]) / s["shots"]) / (s["r_ideal"] - 0.5)


best = {}
for device in FIG3EF_DEVICES:
    candidates = sorted(((score(by)[0], chain.qubits, chain) for chain, by in sweeps[device].items()
                         if SCORE_DEPTH in by and len(by) >= 3), reverse=True)
    value, _, chain = candidates[0]
    sigma_score = score(sweeps[device][chain])[1]
    ties = sum(1 for other, _, _ in candidates[1:] if other > value - sigma_score)
    best[device] = dict(chain=chain, by=sweeps[device][chain], score=value, sigma=sigma_score, ties=ties)

fig, ax = plt.subplots(figsize=(3, 5))
all_p = sorted({p for b in best.values() for p in b["by"]})
fine = np.arange(min(all_p), max(all_p) + 1)
ax.plot(fine, [noiseless(p) for p in fine], "--", lw=2, color="black", zorder=2, label="noiseless")
ax.axhspan(0.5 - 3 * sigma, 0.5 + 3 * sigma, color="gray", alpha=0.25, lw=0, zorder=0, label="random $3\\sigma$")
for device, b in best.items():
    ps = sorted(b["by"])
    ax.plot(ps, [b["by"][p]["r"] for p in ps], "o-", ms=9, lw=2.4, markeredgecolor="black",
            color=COLOURS[device], zorder=3, label=device.replace("_", " "))
ax.set_xticks([3, 6, 9, 12])
ax.set_ylim(0.4, 1.02)
if SHOW_LABELS:
    ax.set(xlabel="$p$", ylabel="approximation ratio $r$")
    ax.legend(fontsize=10, loc="lower left")
fig.savefig(FIGURES / "fig3e_mcm_best_triplet_r.pdf", transparent=True, bbox_inches="tight")
plt.show()

print(f"best triplet = highest r_ovl at p = {SCORE_DEPTH}\n")
print(f"{'device':>13} {'triplet (d1, a, d2)':>20} {'r_ovl':>6} {'sigma':>6} {'ties':>5}   r by depth")
for device, b in best.items():
    print(f"{device:>13} {str(b['chain'].qubits):>20} {b['score']:>6.3f} {b['sigma']:>6.3f} {b['ties']:>5}   "
          + "  ".join(f"p{p}={b['by'][p]['r']:.3f}" for p in sorted(b["by"])))

## 4. Fig. 3f - the best triplets as normalised quality

**What it shows.** The triplets of Fig. 3e as $r_{\rm ovl} = (r - r_{\rm rand})/(r_{\rm ideal} - r_{\rm rand})$, which
divides out the ramp: a noiseless device sits on the dashed line at 1 at every depth, and 0 is random
guessing. The decay of each curve is what the measure-and-correct cycle costs as layers accumulate.

**Significance.** A point is drawn as a filled circle only if $r > r_{\rm rand} + 3\sigma_{\rm rand}$ for that run's
shots, i.e. significantly better than random sampling. A point that fails this is drawn as an ✕ on a dashed
continuation, so the sweep stays visible without being read as a measurement. The grey region is the
same threshold in $r_{\rm ovl}$ units, $3\sigma_{\rm rand} / (r_{\rm ideal}(p) - 1/2)$, which shrinks with depth as
$r_{\rm ideal}$ grows. Anything inside it cannot be told apart from random.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 5))
failed = []
for device, b in best.items():
    colour, ps = COLOURS[device], sorted(b["by"])
    ys = np.array([b["by"][p]["r_ovl"] for p in ps])
    ok = np.array([b["by"][p]["r"] > 0.5 + 3 * random_sigma(b["by"][p]["shots"]) for p in ps])
    failed += [(device, p, b["by"][p]["r"], 0.5 + 3 * random_sigma(b["by"][p]["shots"]))
               for p, passed in zip(ps, ok) if not passed]
    ps = np.array(ps)
    if not ok.all():
        ax.plot(ps, ys, "--", lw=1.6, alpha=0.65, color=colour, zorder=2)
    ax.plot(ps[ok], ys[ok], "o-", ms=9, lw=2.4, markeredgecolor="black", color=colour, zorder=3,
            label=device.replace("_", " "))
    ax.plot(ps[~ok], ys[~ok], "x", ms=11, mew=3, color=colour, zorder=4)

fine = np.arange(3, 13)
threshold = 3 * sigma / (np.array([noiseless(p) for p in fine]) - 0.5)
ax.fill_between(fine, 0, threshold, color="gray", alpha=0.25, lw=0, zorder=0)
ax.axhline(1.0, ls="--", lw=2, color="black", zorder=1)
ax.set_xticks([3, 6, 9, 12])
ax.set_ylim(0, 1.08)
if SHOW_LABELS:
    ax.set(xlabel="$p$", ylabel=r"$r_{\rm ovl}$")
    ax.legend(fontsize=10, loc="lower left")
fig.tight_layout()
fig.savefig(FIGURES / "fig3f_mcm_best_triplet_rovl.pdf", transparent=True, bbox_inches="tight")
plt.show()

for device, p, r, limit in failed:
    print(f"✕ {device} p={p}: r = {r:.3f} <= random + 3 sigma = {limit:.3f} (drawn, but not a measurement)")
print(f"random threshold: r > {0.5 + 3 * sigma:.3f}; in r_ovl {threshold[0]:.3f} at p=3 down to {threshold[-1]:.3f} at p=12")

## 5. Fig. 4 - effective error per edge against chain length: Helios-1 and IBM Boston

**What it shows.** For MCM chains of $N_q$ spins, the effective error per edge $\lambda_{\rm eff}$: the two-qubit
depolarizing strength per edge interaction that reproduces the measured decay,

$$r_{\rm ovl}(p) = 2^{-(\kappa_0/N_q)\,(N_q-1)\,p\,\lambda_{\rm eff}} ,$$

with $\kappa_0$ from the depolarizing simulation of `noise_study.ipynb` (`data/noise_study/kappa_fits.json`).
Squares are Helios-1 hardware (50 shots), circles IBM Boston (1000 shots). Lower is better. A flat curve means
each edge costs the same however long the chain, and a rising one means longer chains cost more per edge.

**Why the serialized Helios-1 runs are worse.** Every Helios-1 run before September (grey: 9 March at
$N_q = 40$, 25 May at $N_q = 30$, 26 May at $N_q = 50$; purple: 25 August at $N_q = 30, 40, 50$) built each layer
with the serial program `qaoa_mcm`. It emits the $N_q - 1$ gadgets bond by bond, and gadget $i+1$ shares a data
qubit with gadget $i$, so it has to wait for gadget $i$'s measurement and correction. A layer is then a chain of
$N_q - 1$ dependent measure-and-correct steps. The data qubits sit idle through all of them, so the memory error
per layer grows with the chain length, and the fit charges it to every edge. The 9 September runs (orange) use
the parallel program `qaoa_mcm_parallel` (the program `benchmark_quantinuum.ipynb` submits in this repository).
It runs the even bonds, then the odd bonds, with the gadgets of a class on disjoint qubits, so none waits on its
neighbour, and at most 8 run at once in Helios-1's operation zones.

The 25 August and 9 September campaigns run the same chains, depths and shots, two weeks apart, so the step
between them isolates the ordering: $\lambda_{\rm eff}$ drops 3.1x, 2.5x and 3.4x at $N_q = 30, 40, 50$ (printed
below). The grey curve is serialized too, so the further ~2x between it and 25 August (1.9x, 1.9x, 2.3x) is not
the ordering. Those runs are three to five months apart, and the gap most likely reflects improvements in the
machine's calibration over that time (no calibration record of these runs is kept to confirm it).

*Provenance of the orderings.* The serial program is what the Quantinuum notebook of the earlier repository
contained for the May runs (commit of 26 May 2026, whose saved output is the 25 May run). The 25 August runs were
serialized as well; the fit files of that repository record them as "colour?", which is wrong. The parallel program
first appears in the commit of 1 September. The 9 September runs are recorded as the two-colour ordering, and the
15 September run is known to use it.

**IBM Boston.** On 25 August each size used its own layout chosen at submission. The 27 and 31 August
campaigns are pinned to the same chain of physical qubits, so the gap between them is run-to-run
repeatability. All three use the two-colour ordering.

**Fits.** Helios-1 over $p = 3, 5, 10$ (all it ran), Boston over $p = 10\ldots50$, where its decay is closest to
a pure exponential. Each device is therefore compared only with itself. Points at or below random are
dropped. Fitted values only, no error bars: with 3-6 points per fit the curve-fit covariance reflects the
number of points more than the measurement.

| series | runs in `data/results/<backend>/chain/mcm/` |
|---|---|
| Helios-1, earlier (serial) | `20260309_0726` ($N_q = 40$, 20 shots), `20260525_0955` (30), `20260526_0735` (50) |
| Helios-1, 08-25 (serial) | `20260825_1505` (30), `20260825_1540` + `_1541` (40), `20260825_1542` + `_1543` + `_1606` (50) |
| Helios-1, 09-09 (parallel) | `20260909_1423` (30), `20260909_1509` (40), `20260909_1521` (50) |
| ibm_boston, 08-25 | `20260825_1618` (10), `_1619` (20), `_1620` (30, 40), `_1621` (50), `_1629` (60) |
| ibm_boston, 08-27 | `20260827_1035` ($N_q = 10\ldots60$) |
| ibm_boston, 08-31 | `20260831_1122` ($N_q = 10\ldots60$) |

**Difference from the first version of the figure.** Every $\lambda_{\rm eff}$ is 0.95x the published value,
the ratio of the two $\kappa_0$: 3.69 from the per-edge simulation here, against 3.51 fitted earlier with a
per-CNOT error. The references are exact here, and at these sizes that moves no value by more than 2%.
The shapes and the ordering of the curves are unchanged.

In [ ]:
import json

from qecbench.noise_study import fit_lambda_eff

KAPPA_0 = json.loads((ROOT / "data" / "noise_study" / "kappa_fits.json").read_text())["kappa_0"]["value"]
HELIOS_WINDOW, BOSTON_WINDOW = [3, 5, 10], [10, 15, 20, 30, 40, 50]
FIG4_SERIES = [   # legend, backend, run stamps, fit window, colour, marker, linestyle
    ("Helios-1, earlier (serial)", "Helios-1", ["20260309_0726", "20260525_0955", "20260526_0735"],
     HELIOS_WINDOW, "grey", "s", "--"),
    ("Helios-1, 08-25 (serial)", "Helios-1", ["20260825_1505", "20260825_1540", "20260825_1541", "20260825_1542",
                                     "20260825_1543", "20260825_1606"], HELIOS_WINDOW, "tab:purple", "s", "-"),
    ("Helios-1, 09-09 (parallel)", "Helios-1", ["20260909_1423", "20260909_1509", "20260909_1521"],
     HELIOS_WINDOW, "tab:orange", "s", "-"),
    ("ibm_boston, 08-25", "ibm_boston", ["20260825_1618", "20260825_1619", "20260825_1620", "20260825_1621",
                                         "20260825_1629"], BOSTON_WINDOW, "tab:green", "o", "-"),
    ("ibm_boston, 08-27", "ibm_boston", ["20260827_1035"], BOSTON_WINDOW, "tab:blue", "o", "-"),
    ("ibm_boston, 08-31", "ibm_boston", ["20260831_1122"], BOSTON_WINDOW, "tab:red", "o", "-"),
]

lambda_by_nq = {}
for legend, backend, stamps, window, *_ in FIG4_SERIES:
    fits = {}
    for chain, by_depth in runs(backend, stamps, kind="mcm").items():
        n = chain.n_data
        ps = np.array([p for p in window if p in by_depth], dtype=float)
        ovl = np.array([by_depth[int(p)]["r_ovl"] for p in ps])
        keep = ovl > 0
        if keep.sum() >= 2:
            fits[n] = fit_lambda_eff(ps[keep], ovl[keep], n - 1, KAPPA_0 / n)["lambda_eff"]
    lambda_by_nq[legend] = dict(sorted(fits.items()))

fig, ax = plt.subplots(figsize=(4, 5))
for legend, _, _, _, colour, marker, ls in FIG4_SERIES:
    fits = lambda_by_nq[legend]
    ax.plot(list(fits), list(fits.values()), marker=marker, color=colour, linestyle=ls, markersize=8,
            markeredgecolor="black", label=legend)
ax.set_xlabel(r"$N_q$", fontsize=14)
ax.set_ylabel(r"$\lambda_{eff}$", fontsize=14)
ax.set_yticks([0.0, 0.01, 0.02, 0.03, 0.04])
ax.set_xticks(sorted({n for fits in lambda_by_nq.values() for n in fits}))
ax.legend(fontsize=10, loc="upper center", bbox_to_anchor=(0.4, 1.35), ncols=2)
plt.tight_layout()
fig.savefig(FIGURES / "fig4_lambda_eff_old_vs_new.pdf", transparent=True, bbox_inches="tight")
plt.show()

sizes = sorted({n for fits in lambda_by_nq.values() for n in fits})
print(f"kappa_0 = {KAPPA_0:.3f}; lambda_eff per edge\n")
print(f"{'':<28}" + "".join(f"{n:>11}" for n in sizes))
for legend, fits in lambda_by_nq.items():
    print(f"{legend:<28}" + "".join(f"{fits[n]:>11.3e}" if n in fits else f"{'-':>11}" for n in sizes))
print()
for legend, fits in lambda_by_nq.items():
    lo, hi = min(fits), max(fits)
    print(f"{legend:<28} grows {fits[hi] / fits[lo]:.1f}x from N_q = {lo} to {hi}")
earlier, august, parallel = (lambda_by_nq[k] for k in ("Helios-1, earlier (serial)", "Helios-1, 08-25 (serial)",
                                                        "Helios-1, 09-09 (parallel)"))
print("\nHelios-1, the ordering (08-25 serial / 09-09 parallel):  "
      + ", ".join(f"N_q = {n}: {august[n] / parallel[n]:.1f}x" for n in august if n in parallel))
print("Helios-1, calibration over time (earlier serial / 08-25 serial): "
      + ", ".join(f"N_q = {n}: {earlier[n] / august[n]:.1f}x" for n in earlier if n in august))

## 6. Fig. 5 - the noise model, and effective errors from it

Fig. 5 turns measured overlaps into an effective error per edge, $\lambda_{\rm eff}$. Panels a-c are redrawn
from `notebooks/noise_study.ipynb` (sections 4, 6 and 7): the model, its calibration on a 10-spin chain, and
$\lambda_{\rm eff}$ across devices and chain lengths. Panel d sets the devices against their release dates.

**The model.** Put a two-qubit depolarizing channel of strength $\lambda$ after every edge interaction of an
$N_q$-spin chain run at depth $p$. The overlap then falls on

$$r_{\rm ovl} = 2^{-\kappa_r\,\varepsilon_{\rm acc}},\qquad \varepsilon_{\rm acc} = N_{\rm edges}\,p\,\lambda,\qquad \kappa_r = \kappa_0/N_q ,$$

with $N_{\rm edges} = N_q - 1$. Inverting it on a device's overlaps, with $\kappa$ held fixed, gives
$\lambda_{\rm eff}$. The simulation behind $\kappa_0$ is exact in expectation and uses no density matrix; its
tables are stored in `data/noise_study/chain_depolarizing.json`, so these panels need no simulation.

### Fig. 5a - $\kappa_r$ for each chain length, and $\kappa_0$

Left: the approximation-ratio overlap against $\varepsilon_{\rm acc}$, one colour per $N_q = 5\ldots15$, all
depths ($p = 2, 5, 10$) together. Every depth of a size lands on one curve, and the lines are the
per-size fits $2^{-\kappa_r\varepsilon_{\rm acc}}$. Right: the fitted $\kappa_r$ against $N_q$ ($3\sigma$ error bars)
and the one-parameter law $\kappa_r = \kappa_0/N_q$.

In [ ]:
import json

from qecbench.noise_study import (accumulated_error, collapse, device_overlaps, device_run, fit_kappa,
                                  fit_kappa0, fit_lambda_eff, load_study, overlap_model)

STUDY = load_study(ROOT / "data" / "noise_study" / "chain_depolarizing.json")
NQS, STUDY_DEPTHS, LAMBDAS = list(range(5, 16)), [2, 5, 10], np.logspace(-5, 0, 25)
study = {key: d for key, d in STUDY.items() if key[1] in STUDY_DEPTHS}

fits_r = {n: fit_kappa(*collapse(study, n, LAMBDAS, "r")[:2]) for n in NQS}
KAPPA_0, kappa0_err, _ = fit_kappa0(NQS, [fits_r[n]["kappa"] for n in NQS])

cmap = plt.get_cmap("jet")
size_colour = {n: cmap(i / max(len(NQS) - 1, 1)) for i, n in enumerate(NQS)}
fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
x = np.logspace(-2, 2, 300)
for n in NQS:
    eps, ovl, err, depth, tail = collapse(study, n, LAMBDAS, "r")
    keep = (eps >= 1e-2) & (eps <= 1e2)
    ax.errorbar(eps[keep], ovl[keep], yerr=err[keep], fmt="o", ms=3.5, color=size_colour[n], alpha=0.75,
                label=f"$N_q$={n}" if n % 2 else None)
    ax.plot(x, overlap_model(x, fits_r[n]["kappa"]), color=size_colour[n], lw=1)
ax.set(xscale="log", xlim=(1e-2, 1e2), ylim=(-0.05, 1.05), xlabel=r"$\varepsilon_{acc} = N_{edges}\,p\,\lambda$",
       ylabel=r"$r_{\rm ovl}$")
ax.legend(fontsize=8, ncol=2)
dense = np.linspace(min(NQS) - 0.5, max(NQS) + 0.5, 200)
for n in NQS:
    bx.errorbar(n, fits_r[n]["kappa"], yerr=3 * fits_r[n]["stderr"], fmt="o", color=size_colour[n], capsize=3)
bx.plot(dense, KAPPA_0 / dense, ":", color="purple", lw=2, label=fr"$\kappa_0/N_q$, $\kappa_0$={KAPPA_0:.2f}")
bx.set(yscale="log", xlabel="$N_q$", ylabel=r"$\kappa_r$")
bx.legend()
fig.tight_layout()
fig.savefig(FIGURES / "fig5a_kappa_r_scaling.pdf", transparent=True, bbox_inches="tight")
plt.show()
print(f"kappa_0 = {KAPPA_0:.3f} ± {kappa0_err:.3f}   (kappa_r = kappa_0 / N_q)")

### Fig. 5b - $\lambda_{\rm eff}$ of a 10-spin chain on IBM Boston and Helios-1E

Grey: the depolarizing simulation at $N_q = 10$ over the depths the devices ran ($p = 10\ldots50$), and its
fit. Coloured: device overlaps, each placed at $\varepsilon_{\rm acc} = 9\,p\,\lambda_{\rm eff}$ with its fitted
$\lambda_{\rm eff}$; circles are the direct circuit, triangles MCM. Left, the approximation ratio with
$\kappa_r = \kappa_0/10$; right, the probability of the optimum with $\kappa_\rho$ fitted at these depths.
Points that follow the grey curve degrade like uniform depolarizing noise.

**Data.** `data/results/ibm_boston/chain/{direct,mcm}/`: `20260506_1507` (direct) and `20260506_1537` (MCM),
10 000 shots, $p = 10\ldots50$. `data/results/Helios-1E/chain/{direct,mcm}/`: `20260507_0900` (direct) and
`20260508_0700` (MCM), 1 000 shots on **Quantinuum's Helios-1E emulator** (flagged as simulated),
$p = 1\ldots10, 15, 20, 30, 40, 50$. Overlaps above 1.2 (shot noise on nearly noiseless points) are set to 1.

In [ ]:
DEVICE_N, DEEP_DEPTHS = 10, [10, 20, 30, 40, 50]
FIG5B_RUNS = {   # label -> (backend, kind, run stamps, depths)
    ("ibm_boston", "direct"): ("ibm_boston", "direct", ["20260506_1507"], [10, 15, 20, 30, 40, 50]),
    ("ibm_boston", "MCM"):    ("ibm_boston", "mcm", ["20260506_1537"], [10, 15, 20, 30, 40, 50]),
    ("Helios-1E", "direct"):  ("Helios-1E", "direct", ["20260507_0900"], [*range(1, 11), 15, 20, 30, 40, 50]),
    ("Helios-1E", "MCM"):     ("Helios-1E", "mcm", ["20260508_0700"], [*range(1, 11), 15, 20, 30, 40, 50]),
}
deep = {key: d for key, d in STUDY.items() if key[0] == DEVICE_N and key[1] in DEEP_DEPTHS}
kappa_rho = fit_kappa(*collapse(deep, DEVICE_N, LAMBDAS, "prob")[:2])["kappa"]
kappas = {"r": KAPPA_0 / DEVICE_N, "prob": kappa_rho}
set1 = plt.get_cmap("Set1")
device_colour, marker = {"ibm_boston": set1(0), "Helios-1E": set1(1)}, {"direct": "o", "MCM": "^"}

lambda_5b = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, observable, ylabel in zip(axes, ("r", "prob"), (r"$r_{\rm ovl}$", r"$\rho_{\rm ovl}$")):
    eps, ovl, *_ = collapse(deep, DEVICE_N, LAMBDAS, observable)
    ax.plot(eps, ovl, "o", color="gray", alpha=0.4, ms=4, label="depolarizing simulation")
    ax.plot(x, overlap_model(x, kappas[observable]), "--", color="black", lw=2,
            label=fr"$2^{{-\kappa\,\varepsilon_{{acc}}}}$, $\kappa$={kappas[observable]:.3f}")
    for (device, circuit), (backend, kind, stamps, depths) in FIG5B_RUNS.items():
        ps, values = device_overlaps(device_run(RESULTS, backend, kind, stamps, DEVICE_N, depths), DEVICE_N,
                                     observable=observable)
        fit = fit_lambda_eff(ps, values, DEVICE_N - 1, kappas[observable])
        lambda_5b[(device, circuit, observable)] = fit["lambda_eff"]
        ax.plot(accumulated_error(DEVICE_N - 1, ps, fit["lambda_eff"]), values, marker[circuit], ms=8,
                color=device_colour[device], mec="black", label=f"{device} {circuit}")
    ax.set(xscale="log", xlim=(1e-2, 1e2), ylim=(-0.05, 1.1), xlabel=r"$\varepsilon_{acc}$", ylabel=ylabel)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "fig5b_lambda_eff_nq10.pdf", transparent=True, bbox_inches="tight")
plt.show()

print(f"kappa_r = {kappas['r']:.3f}, kappa_rho = {kappa_rho:.3f} (p = {DEEP_DEPTHS})\n")
print(f"{'':<20}{'lambda_eff from r':>20}{'from rho':>12}")
for device, circuit in FIG5B_RUNS:
    print(f"{device + ' ' + circuit:<20}{lambda_5b[(device, circuit, 'r')]:>20.2e}{lambda_5b[(device, circuit, 'prob')]:>12.2e}")

### Fig. 5c - $\lambda_{\rm eff}$ against chain length, across devices

The per-edge $\lambda_{\rm eff}$ of every device at every chain length it ran, for the direct circuit (circles,
solid lines) and the MCM circuit (crosses, dashed lines), with $\kappa_r = \kappa_0/N_q$ from Fig. 5a. The
chains run to $N_q = 60$, far past the $N_q \le 15$ of the simulation, so the $1/N_q$ law is extrapolated.
Dotted segments are Quantinuum **hardware**, a dotted link joins an emulator point to a hardware point, and the
Helios-1E emulator is left out so that no simulated point looks measured.

| device | direct circuits | MCM circuits |
|---|---|---|
| `ibm_boston`, `ibm_pittsburgh`, `ibm_kingston` | April-May 2026, $N_q = 10\ldots60$ | late August 2026, $N_q = 10\ldots60$ |
| Quantinuum Helios | Helios-1 hardware, $N_q = 30, 40, 50$ | Helios-1 hardware: $N_q = 20$ (15 September), $30, 40, 50$ (9 September) |
| Quantinuum H2 | H2-1E emulator $N_q = 10, 20$; H2-1 hardware $N_q = 30$ | same split, 20 August 2026 |
| `ibm_fez`, `ibm_marrakesh`, `ibm_phoenix` | - | late August / early September 2026 |

**Read the MCM curves against each other.** Every MCM curve here uses the two-colour ordering of the gadgets:
the IBM campaigns, the Helios-1 runs (the parallel program, Fig. 4) and the H2 runs. H2 does not run Guppy, so its
runs were built as pytket circuits (`benchmarking_quantinuum_h2-1.ipynb` in the earlier repository). That builder
colours the bonds into even and odd classes and gives each class a pool of $\lfloor N_q/2 \rfloor$ ancillas, reset and
reused between classes, since H2-1 has a fixed 56-qubit register. Within a class the gadgets touch disjoint data
qubits and disjoint ancillas, so none waits on another. The stored runs carry exactly those pools (5, 10 and 15
ancillas at $N_q = 10, 20, 30$), which confirms the colouring was used. The direct campaigns come from an earlier
period, and no direct runs were made alongside the new MCM ones.

**Fit windows.** MCM: $p = 3, 5, 10$, the only depths every device ran (Helios-1 stops at $p = 10$). This window
inflates $\lambda_{\rm eff}$ by roughly 1.2-1.5x against a $p \ge 10$ window, alike for every device. Direct:
every depth run among $3\ldots50$, after dropping the leading plateau (points up to the maximum overlap). Points
at or below random are dropped. There are no error bars: with 2-8 points per fit the curve-fit covariance
reflects how many points survived, not the measurement.

| device | circuit | runs in `data/results/<backend>/chain/<kind>/` |
|---|---|---|
| `ibm_boston` | direct | `20260512_1429` (10), `20260428_110845` (20), `_110857` (30), `_110905` (40), `_110915` (50), `20260430_0834` (60) |
| | MCM | `20260827_1035` |
| `ibm_pittsburgh` | direct | `20260512_1402` (10), `20260428_111001` (20), `_110953` (30), `_110944` (40), `_110932` (50), `20260430_1618` (60) |
| | MCM | `20260828_1025` |
| `ibm_kingston` | direct | `20260512_1402` (10), `20260505_0806` (20), `20260505_0805` (30-60) |
| | MCM | `20260827_1205` |
| Helios-1 | direct | `20260525_0957` (30), `20260309_0726` (40), `20260526_0802` (50) |
| | MCM | `20260915_1035` (20), `20260909_1423` (30), `_1509` (40), `_1521` (50) |
| H2-1E / H2-1 | direct | `20260820_0646` (H2-1E, 10, 20), `20260820_0651` (H2-1, 30) |
| | MCM | `20260820_0647` (H2-1E, 10, 20), `20260820_0650` (H2-1, 30) |
| `ibm_fez`, `ibm_marrakesh`, `ibm_phoenix` | MCM | `20260828_1028`, `20260828_1034`, `20260904_1435` |

In [ ]:
from qecbench.noise_study import decaying_part

LAMBDA_SERIES = {   # device -> {circuit: [(backend, run stamp), ...]}
    "ibm_boston":        {"direct": [("ibm_boston", s) for s in ("20260512_1429", "20260428_110845", "20260428_110857",
                                                                  "20260428_110905", "20260428_110915", "20260430_0834")],
                          "mcm":    [("ibm_boston", "20260827_1035")]},
    "ibm_pittsburgh":    {"direct": [("ibm_pittsburgh", s) for s in ("20260512_1402", "20260428_111001", "20260428_110953",
                                                                      "20260428_110944", "20260428_110932", "20260430_1618")],
                          "mcm":    [("ibm_pittsburgh", "20260828_1025")]},
    "ibm_kingston":      {"direct": [("ibm_kingston", s) for s in ("20260512_1402", "20260505_0806", "20260505_0805")],
                          "mcm":    [("ibm_kingston", "20260827_1205")]},
    "quantinuum_helios": {"direct": [("Helios-1", s) for s in ("20260525_0957", "20260309_0726", "20260526_0802")],
                          "mcm":    [("Helios-1", s) for s in ("20260915_1035", "20260909_1423", "20260909_1509", "20260909_1521")]},
    "quantinuum_h2":     {"direct": [("H2-1E", "20260820_0646"), ("H2-1", "20260820_0651")],
                          "mcm":    [("H2-1E", "20260820_0647"), ("H2-1", "20260820_0650")]},
    "ibm_fez":           {"mcm": [("ibm_fez", "20260828_1028")]},
    "ibm_marrakesh":     {"mcm": [("ibm_marrakesh", "20260828_1034")]},
    "ibm_phoenix":       {"mcm": [("ibm_phoenix", "20260904_1435")]},
}
LAMBDA_COLOURS = {"ibm_boston": 0, "ibm_pittsburgh": 1, "ibm_kingston": 2, "quantinuum_helios": 3,
                  "quantinuum_h2": 4, "ibm_fez": 6, "ibm_marrakesh": 7, "ibm_phoenix": 8}   # Set1 indices
QUANTINUUM_HARDWARE = {"Helios-1", "H2-1"}                                                  # drawn dotted
LAMBDA_WINDOWS = {"direct": [3, 5, 10, 15, 20, 30, 40, 50], "mcm": [3, 5, 10]}
DROP_PLATEAU = {"direct": True, "mcm": False}


def lambda_vs_nq(backend, stamps, window, kind="mcm", drop_plateau=False):
    # {N_q: fit} with the per-edge model, kappa = kappa_0 / N_q, over the depths in `window`
    out = {}
    for chain, by_depth in runs(backend, stamps, kind=kind).items():
        n = chain.n_data
        ps = np.array([p for p in window if p in by_depth], dtype=float)
        ovl = np.array([by_depth[int(p)]["r_ovl"] for p in ps])
        ps, ovl = ps[ovl > 0], ovl[ovl > 0]
        if drop_plateau:
            ps, ovl = decaying_part(ps, ovl)
        if len(ps) >= 2:
            out[n] = {**fit_lambda_eff(ps, ovl, n - 1, KAPPA_0 / n), "backend": backend}
    return dict(sorted(out.items()))


lambda_by_nq = {}
for device, kinds in LAMBDA_SERIES.items():
    for kind, device_runs in kinds.items():
        stamps_by_backend = {}
        for backend, stamp in device_runs:
            stamps_by_backend.setdefault(backend, []).append(stamp)
        lambda_by_nq[(device, kind)] = {}
        for backend, stamps in stamps_by_backend.items():
            lambda_by_nq[(device, kind)].update(lambda_vs_nq(backend, stamps, LAMBDA_WINDOWS[kind], kind, DROP_PLATEAU[kind]))

colours = plt.get_cmap("Set1")
style = {"direct": dict(marker="o", ls="-", ms=7), "mcm": dict(marker="X", ls="--", ms=8)}
fig, ax = plt.subplots(figsize=(3, 4))
for device, kinds in LAMBDA_SERIES.items():
    colour = colours(LAMBDA_COLOURS[device])
    labelled = False
    for kind in kinds:
        fits = lambda_by_nq[(device, kind)]
        segments = []                               # consecutive points on the same backend
        for n in sorted(fits):
            if segments and segments[-1][0] == fits[n]["backend"]:
                segments[-1][1].append(n)
            else:
                segments.append((fits[n]["backend"], [n]))
        previous = None
        for backend, nqs in segments:
            ys = [fits[n]["lambda_eff"] for n in nqs]
            ax.plot(nqs, ys, color=colour, mec="black", marker=style[kind]["marker"], ms=style[kind]["ms"],
                    ls=":" if backend in QUANTINUUM_HARDWARE else style[kind]["ls"], label=None if labelled else device)
            labelled = True
            if previous is not None:
                ax.plot([previous[0], nqs[0]], [previous[1], ys[0]], color=colour, ls=":")
            previous = (nqs[-1], ys[-1])
ax.plot([], [], "o", color="black", ms=7, markeredgecolor="black", linestyle="None", label="direct")
ax.plot([], [], "X", color="black", ms=8, markeredgecolor="black", linestyle="None", label="MCM")
ax.set_yscale("log")
if SHOW_LABELS:
    ax.set(xlabel="$N_q$", ylabel=r"$\lambda_{\rm eff}$ per edge")
ax.legend(loc="upper right", fontsize=10, bbox_to_anchor=(1.18, 1.42), ncols=3)
fig.savefig(FIGURES / "fig5c_lambda_eff_by_nq.pdf", transparent=True, bbox_inches="tight")
plt.show()

sizes = sorted({n for fits in lambda_by_nq.values() for n in fits})
for kind in ("direct", "mcm"):
    print(f"lambda_eff per edge, {kind}, window p = {LAMBDA_WINDOWS[kind]}"
          + (" (leading plateau dropped)" if DROP_PLATEAU[kind] else ""))
    print(f"{'':<20}" + "".join(f"{n:>10}" for n in sizes))
    for device in LAMBDA_SERIES:
        fits = lambda_by_nq.get((device, kind))
        if fits:
            print(f"{device:<20}" + "".join(f"{fits[n]['lambda_eff']:>10.2e}" if n in fits else f"{'-':>10}" for n in sizes))
    print()

### Fig. 5d - MCM error per edge against when each device became available

For eight devices, the mean MCM $\lambda_{\rm eff}$ over the chain lengths measured (bars: min-max over $N_q$,
the spread across chain length, not a shot-noise error), placed at the device's first public availability.
Shaded bands cover each vendor's range, and the dashed lines are a guide to the eye: a least-squares line
through $\log_{10}\lambda$ against release date, i.e. a constant factor per year. The two vendors are kept as
separate families. Every IBM superconducting device sits about an order of magnitude above Quantinuum's
trapped-ion machines, so one trend through all points would hide the main result. With 6 IBM points over 4
processor families and 2 Quantinuum points, the trends are anecdotes, not measurements.

**Fits.** $p = 3, 5, 10$ for every device (the only depths every device ran; Helios-1 stops at $p = 10$). The
exception is `ibm_phoenix`, which has no $p = 3$ and reaches the random floor by $p \approx 25$, so it uses
$p = 5, 10, 15, 20$. Two caveats flatter Phoenix: only $N_q = 10\ldots40$ could be embedded cleanly (its
$\lambda_{\rm eff}$ grows with $N_q$), and dropping $p = 3$ alone lowers $\lambda$ by about 1.3x.

**Release dates** are metadata, not data: first public availability as announced by the vendor (`precision`
says whether the date is the system's or only its processor family's, and the source of each is listed).

| device | MCM run(s) in `data/results/<device>/chain/mcm/` | $N_q$ |
|---|---|---|
| `ibm_fez` | `20260828_1028` | 10-60 |
| `ibm_marrakesh` | `20260828_1034` | 10-60 |
| `ibm_kingston` | `20260827_1205` | 10-60 |
| `ibm_pittsburgh` | `20260828_1025` | 10-60 |
| `ibm_boston` | `20260827_1035` | 10-60 |
| `ibm_phoenix` | `20260904_1435` | 10-40 |
| `H2-1` | `20260820_0650` | 30 |
| `Helios-1` | `20260909_1423`, `_1509`, `_1521` | 30-50 |

In [ ]:
import datetime

import matplotlib.dates as mdates

WINDOW_5D = [3, 5, 10]
DEVICE_INTRO = {   # device -> run stamps, Set1 colour, first availability, processor, precision, source
    "ibm_fez": dict(stamps=["20260828_1028"], c=6, date="2024-07-01", processor="Heron r2", precision="month",
                    source="https://qpustatus.com/ibm-fez"),
    "ibm_marrakesh": dict(stamps=["20260828_1034"], c=7, date="2024-11-13", processor="Heron r2", precision="system",
                          source="https://qpustatus.com/ibm-marrakesh"),
    "ibm_kingston": dict(stamps=["20260827_1205"], c=2, date="2025-04-09", processor="Heron r2", precision="system",
                         source="https://quantum.cloud.ibm.com/announcements/product-updates/2025-04-09-aachen-kingston"),
    "ibm_pittsburgh": dict(stamps=["20260828_1025"], c=1, date="2025-07-31", processor="Heron r3", precision="system",
                           source="https://quantum.cloud.ibm.com/announcements/en/product-updates/2025-07-31-pittsburgh-sherbrooke"),
    "ibm_boston": dict(stamps=["20260827_1035"], c=0, date="2025-11-12", processor="Heron r3", precision="system",
                       note="access expanded 2026-01", source="https://research.ibm.com/events/qdc-2025"),
    "H2-1": dict(stamps=["20260820_0650"], c=4, date="2024-06-05", processor="Quantinuum H2 (56q)", precision="system",
                 source="https://www.quantinuum.com/press-releases/quantinuum-launches-industry-first-trapped-ion-56-qubit-quantum-computer-that-challenges-the-worlds-best-supercomputers"),
    "ibm_phoenix": dict(stamps=["20260904_1435"], c=8, date="2026-09-02", processor="Nighthawk r2", precision="system",
                        window=[5, 10, 15, 20],
                        source="https://quantumcomputingreport.com/ibm-releases-nighthawk-r2-qpu-featuring-active-dissipative-qubit-reset-and-25x-circuit-throughput/"),
    "Helios-1": dict(stamps=["20260909_1423", "20260909_1509", "20260909_1521"], c=3, date="2025-11-05",
                     processor="Quantinuum Helios", precision="system",
                     source="https://thequantuminsider.com/2025/11/05/quantinuum-announces-commercial-launch-of-helios-a-quantum-computer-with-accuracy-to-enable-generative-quantum-ai/"),
}
VENDORS = {"IBM": ("IBM (superconducting)", "tab:blue"), "Quantinuum": ("Quantinuum (trapped ion)", "tab:purple")}

rows = []
for device, info in DEVICE_INTRO.items():
    window = info.get("window", WINDOW_5D)
    values = np.array([f["lambda_eff"] for f in lambda_vs_nq(device, info["stamps"], window).values()])
    rows.append(dict(dev=device, vendor="IBM" if device.startswith("ibm_") else "Quantinuum",
                     date=datetime.date.fromisoformat(info["date"]), mean=float(values.mean()),
                     lo=float(values.min()), hi=float(values.max()), n=len(values), window=window,
                     **{k: v for k, v in info.items() if k not in ("date", "window")}))
rows.sort(key=lambda r: r["date"])

fig, ax = plt.subplots(figsize=(4, 5))
trend = {}
for vendor, (label, colour) in VENDORS.items():
    group = [r for r in rows if r["vendor"] == vendor]
    ax.axhspan(min(r["lo"] for r in group), max(r["hi"] for r in group), color=colour, alpha=0.08, lw=0, zorder=0)
    years = np.array([r["date"].toordinal() / 365.25 for r in group])
    log_lambda = np.log10([r["mean"] for r in group])
    slope, intercept = np.polyfit(years, log_lambda, 1)
    residual = log_lambda - (slope * years + intercept)
    total = float(((log_lambda - log_lambda.mean()) ** 2).sum())
    trend[vendor] = dict(per_year=10 ** slope, r2=1 - float((residual ** 2).sum()) / total if total > 0 else np.nan,
                         n=len(group))
    span = np.linspace(years.min() - 0.15, years.max() + 0.15, 50)
    ax.plot([datetime.date.fromordinal(int(v * 365.25)) for v in span], 10 ** (slope * span + intercept),
            "--", color=colour, lw=1.6, alpha=0.75, zorder=1)
set1 = plt.get_cmap("Set1")
for r in rows:
    ax.errorbar(r["date"], r["mean"], yerr=[[r["mean"] - r["lo"]], [r["hi"] - r["mean"]]], marker="X",
                color=set1(r["c"]), markersize=9, markeredgecolor="black", capsize=4, elinewidth=1.2, capthick=1.2)
ax.set_yscale("log")
low, high = ax.get_ylim()
ax.set_ylim(low / 1.25, high * 1.25)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate(rotation=40)
ax.grid(alpha=0.25, which="both")
if SHOW_LABELS:
    ax.set(xlabel="first available", ylabel=r"mean $\lambda_{eff}$ per edge (MCM)")
plt.tight_layout()
fig.savefig(FIGURES / "fig5d_mcm_error_vs_device_intro.pdf", transparent=True, bbox_inches="tight")
plt.show()

geo = {v: float(np.exp(np.mean(np.log([r["mean"] for r in rows if r["vendor"] == v])))) for v in VENDORS}
print(f"geometric mean lambda_eff (MCM): IBM {geo['IBM']:.3e}, Quantinuum {geo['Quantinuum']:.3e} "
      f"-> IBM is {geo['IBM'] / geo['Quantinuum']:.1f}x higher")
for vendor, t in trend.items():
    print(f"   {vendor:<11} x{t['per_year']:.2f} per year, R2 = {t['r2']:.2f}, n = {t['n']}"
          + ("   <- 2 points, the slope is not a measurement" if t["n"] <= 2 else ""))
print(f"\n{'device':<16}{'first available':<17}{'processor':<22}{'precision':<11}{'window':<16}{'mean lambda':>12}"
      f"{'min - max':>24}{'N_q':>5}")
for r in rows:
    print(f"{r['dev']:<16}{str(r['date']):<17}{r['processor']:<22}{r['precision']:<11}{str(r['window']):<16}"
          f"{r['mean']:>12.3e}{r['lo']:>12.3e} - {r['hi']:.3e}{r['n']:>5}")
print("\nsources:")
for r in rows:
    print(f"  {r['dev']:<16}{r['source']}")